## Step 1: Mount Google Drive & Check GPU

In [ ]:
from google.colab import drive
import torch

# Mount Google Drive
drive.mount('/content/drive')

# Check GPU
print("\n" + "="*60)
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"✓ GPU detected: {gpu_name}")
    print(f"  Memory: {gpu_memory:.1f} GB")
else:
    print("⚠ WARNING: No GPU detected!")
    print("  Go to: Runtime → Change runtime type → GPU (T4)")
print("="*60)

## Step 2: Clone Repository

In [ ]:
# Clone the repository (base branch with Phase 2.2)
!git clone https://github.com/ketsiambaku/rna-motif-classification.git
%cd rna-motif-classification
!git checkout base

print("\n✓ Repository cloned and switched to base branch (Phase 2.2)")

## Step 3: Extract Dataset from Google Drive (ZIP)

In [ ]:
import os
import zipfile
from pathlib import Path

# Path to your dataset in Google Drive (now using .zip)
dataset_path = '/content/drive/MyDrive/dataset2.zip'

print("="*60)
if os.path.exists(dataset_path):
    print(f"✓ Found dataset at: {dataset_path}")
    print("  Extracting... (this may take 2-3 minutes)")
    
    try:
        with zipfile.ZipFile(dataset_path, 'r') as zip_ref:
            zip_ref.extractall('.')
        
        # Verify extraction
        if Path('dataset2').exists():
            num_files = sum(1 for _ in Path('dataset2').rglob('*.mrc'))
            print(f"\n✓ Dataset extracted successfully!")
            print(f"  Found {num_files} .mrc files in dataset2/")
        else:
            print("\n⚠ WARNING: dataset2/ folder not found after extraction")
    except Exception as e:
        print(f"\n⚠ ERROR: {e}")
else:
    print(f"\n⚠ ERROR: Dataset not found at: {dataset_path}")
    print("\nPlease upload dataset2.zip to your Google Drive:")
    print("  1. Go to drive.google.com")
    print("  2. Upload dataset2.zip to 'My Drive'")
    print("  3. Re-run this cell")
print("="*60)

## Step 4: Install Dependencies

In [ ]:
# Install required packages
!pip install -q mrcfile biopython scikit-learn

print("✓ Dependencies installed: mrcfile, biopython, scikit-learn")

## Step 5: Train Phase 2.2 Model (Density + Sequence + Pairing)

This cell trains the **3-branch hybrid model** with:
- **3 feature types**: Cryo-EM density + 24 sequence features + 10 base pairing features
- **6 consolidated classes** (small/large internal, bulge, hairpin)
- **Full dataset** (100% of 28,738 samples = 20K train, 6K val, 3K test)
- **Batch size 32** (optimal for T4 GPU)
- **Max epochs: 100** with **early stopping patience: 7**

Expected training time: **20-30 minutes** on T4 GPU

The script will automatically:
1. Train on train set (70%)
2. Validate on val set (20%)
3. Stop early if no improvement for 7 epochs
4. Evaluate best model on test set (10%) at the end

In [ ]:
import subprocess
import sys

# Training command for Phase 2.2 (3-branch model with pairing features)
cmd = [
    sys.executable, 'src/train_hybrid.py',
    '--consolidate',        # Enable 6-class mode
    '--use-subset', '1.0',  # Use 100% of dataset (28,738 samples)
    '--batch-size', '32',   # Optimal for T4 GPU
    '--device', 'cuda',
    '--num-workers', '2'
]

print("="*80)
print("Phase 2.2: Training 3-branch model (Density + Sequence + Pairing)")
print("="*80)
print("\nConfiguration:")
print("  Model: HybridUNet (3-branch architecture)")
print("  Features: Density + 24 sequence + 10 pairing = 3 branches")
print("  Dataset: 100% (28,738 samples)")
print("  Train/Val/Test: 70%/20%/10%")
print("  Batch size: 32")
print("  Max epochs: 100")
print("  Early stopping: patience=7")
print("  Device: CUDA (T4 GPU)")
print("\nCommand:", ' '.join(cmd))
print("="*80 + "\n")

# Run with real-time output
process = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

for line in process.stdout:
    print(line, end='')

process.wait()

print("\n" + "="*80)
if process.returncode == 0:
    print("✓ Training completed successfully!")
    print("✓ Best model saved with validation and test set evaluation")
else:
    print(f"⚠ Training exited with code: {process.returncode}")
print("="*80)

## Step 6: Download Results

## Step 6: Download Training Results to Google Drive

In [ ]:
import shutil
import glob
from pathlib import Path

# Find the most recent experiment directory
exp_dirs = sorted(Path('experiments').glob('hybrid_phase2_1_6class_*'))

if exp_dirs:
    latest_exp = exp_dirs[-1]
    print(f"Found experiment: {latest_exp.name}")
    
    # Copy to Google Drive
    drive_path = Path('/content/drive/MyDrive/rna-motif-results')
    drive_path.mkdir(exist_ok=True)
    
    dest_path = drive_path / latest_exp.name
    shutil.copytree(latest_exp, dest_path, dirs_exist_ok=True)
    
    print(f"\n✓ Results copied to Google Drive:")
    print(f"  {dest_path}")
    print(f"\nSaved files:")
    for f in sorted(dest_path.glob('*')):
        print(f"  - {f.name}")
    
    # Display key results
    import json
    
    if (dest_path / 'config.json').exists():
        with open(dest_path / 'config.json') as f:
            config = json.load(f)
        print(f"\n" + "="*60)
        print("Configuration:")
        for k, v in config.items():
            print(f"  {k}: {v}")
    
    if (dest_path / 'final_val_metrics.json').exists():
        with open(dest_path / 'final_val_metrics.json') as f:
            val_metrics = json.load(f)
        print(f"\n" + "="*60)
        print("Validation Results (Best Model):")
        print(f"  Macro Precision: {val_metrics['macro_precision']:.4f}")
        print(f"  Macro Recall:    {val_metrics['macro_recall']:.4f}")
        print(f"  Macro F1:        {val_metrics['macro_f1']:.4f}")
    
    if (dest_path / 'final_test_metrics.json').exists():
        with open(dest_path / 'final_test_metrics.json') as f:
            test_metrics = json.load(f)
        print(f"\n" + "="*60)
        print("Test Set Results (Best Model):")
        print(f"  Macro Precision: {test_metrics['macro_precision']:.4f}")
        print(f"  Macro Recall:    {test_metrics['macro_recall']:.4f}")
        print(f"  Macro F1:        {test_metrics['macro_f1']:.4f}")
        print("="*60)
else:
    print("⚠ No experiment directories found in experiments/")

In [ ]:
from google.colab import files
import shutil
import os

# Find the latest experiment folder
try:
    exp_folders = [f for f in os.listdir('experiments') if f.startswith('hybrid_phase2_1')]
    if exp_folders:
        exp_dir = max(exp_folders, key=lambda x: os.path.getmtime(os.path.join('experiments', x)))
        
        print(f"Creating zip file for: {exp_dir}")
        shutil.make_archive(f'results_{exp_dir}', 'zip', f'experiments/{exp_dir}')
        
        print(f"Downloading results...")
        files.download(f'results_{exp_dir}.zip')
        print(f"\n✓ Downloaded: results_{exp_dir}.zip")
    else:
        print("No experiment folders found. Did training complete successfully?")
except Exception as e:
    print(f"Error: {e}")

## Optional: Train 15-Class Model for Comparison

Run this cell to train the original 15-class model for comparison.

In [ ]:
import subprocess
import sys

# Training command WITHOUT consolidation (15 classes)
cmd = [
    sys.executable, 'src/train_hybrid.py',
    '--use-subset', '1.0',
    '--batch-size', '32',
    '--device', 'cuda',
    '--num-workers', '2'
]

print("="*80)
print("Starting training with 15 original classes...")
print("="*80 + "\n")

process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in process.stdout:
    print(line, end='')
process.wait()

print("\n" + "="*80)
print(f"Training completed with exit code: {process.returncode}")
print("="*80)

## Optional: Quick Test Run (5% subset)

Use this to test the Phase 2.2 pipeline before running full training.
Should complete in ~3-5 minutes on T4 GPU.

In [ ]:
import subprocess
import sys

cmd = [
    sys.executable, 'src/train_hybrid.py',
    '--consolidate',
    '--use-subset', '0.05',  # Only 5% of data (~1,400 samples)
    '--batch-size', '32',
    '--device', 'cuda',
    '--num-workers', '2'
]

print("="*80)
print("Running quick test with 5% of data (~1,400 samples)...")
print("This will test the Phase 2.2 pipeline end-to-end")
print("="*80 + "\n")

process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in process.stdout:
    print(line, end='')
process.wait()

print("\n" + "="*80)
if process.returncode == 0:
    print("✓ Test run completed! Pipeline is working correctly.")
    print("✓ Ready for full training (100% dataset)")
else:
    print(f"⚠ Test failed with exit code: {process.returncode}")
print("="*80)